Вы работаете аналитиком данных в онлайн-кинотеатре *SkyCinema*.\
Сегодня вы будете работать с **ивентами** (event - *событие*). Ивенты - это записи о клиентских событиях, таких как включение плеера с видео или заход на сайт онлайн-кинотеатра. Одной из важнейших задач аналитики является изучение и правка ошибочных ивентов.

Ваша задача - изучить датасет на наличие ошибок.

Импортируйте файл *event_list.xlsx* и сохраните его в датафрейм.

**Описание данных**

    - account_id - идентификатор пользователя
    - server_time - время совершения действия
    - action_id - идентификатор (тег) действия
    - device_type - тип устройства
    - user_browser - браузер
    - screen_type - тип страницы, на которой совершается действие.
*В рамках данного технического задания вам необходимо взять только те строки, у которых колонка screen_type принимает значение "player".*
    
**Описание ивентов**

    - 'screenview' - заход на страницу с проигрывателем тайтла
    - 'click' - клик по плееру
    - 'playback_start' - включение проигрывателя
    - 'playback_pause' - постановка проигрывателя на паузу
    - 'playback_unpause' - снятия проигрывателя с паузы
    - 'playback_stop'  - выключение страницы с проигрывателем

In [3]:
import pandas as pd

import numpy as np

### Задача 1

Выделите 10 самых активных пользователей (тех, что сделали больше всего действий за данный период времени).\
Каким браузером чаще всего пользовались эти 10 пользователей?\
Какое действие занимает наибольший процент среди всех действий этих 10 пользователей?

In [4]:
event_list = pd.read_excel('event_list.xlsx')
player_screen = event_list[event_list['screen_type'] == 'player']
player_screen.head(3)

,account_id,server_time,screen_type,action_id,device_type,user_browser
33,102313,2021-10-01 09:24:12,player,playback_start,desktop_web,Chrome
34,102313,2021-10-01 09:24:13,player,playback_pause,desktop_web,Chrome
43,102313,2021-10-01 11:53:54,player,playback_start,desktop_web,Chrome


In [13]:
df_gr = player_screen.groupby('account_id').agg(cnt_user= ('action_id', 'count')).reset_index()\
.sort_values('cnt_user', ascending=False).head(10)
df_gr_top10 = df_gr['account_id']

df_browser = player_screen[player_screen['account_id'].isin(df_gr_top10)]\
.groupby('user_browser').agg(cnt_browser = ('user_browser', 'count')).reset_index()\
.sort_values('cnt_browser', ascending=False).head(1)

browser = df_browser['user_browser'].to_list()
print(f'Чаще всего пользовались браузером {browser[0]}')

Чаще всего пользовались браузером Yandex


In [15]:
df_action = player_screen[player_screen['account_id'].isin(df_gr_top10)]\
.groupby('action_id').agg(cnt_action = ('action_id', 'count')).reset_index()\
.sort_values('cnt_action', ascending=False)

per = df_action['cnt_action'].max()/df_action['cnt_action'].sum()*100
action = df_action['action_id'].head(1).to_list()
print(f'Чаще всего использовали действие - {action[0]}, {round(per,2)}%')

Чаще всего использовали действие - click, 62.75%


### Задача 2

Проверьте данные на состоятельность:

1. Есть ли клиенты, которые снимали видео с паузы хотя бы раз, но при этом не ставили видео на паузу ни разу?
2. Есть ли клиенты, которые ставили видео на паузу хотя бы раз, но при этом не включали проигрыватель?

С каких устройств и браузеров заходят пользователи, у которых встречаются подобные аномалии?

*Подсказка*\
Воспользуйтесь методом серии unique(), чтобы для каждого действия (ивента) создать списки с уникальными пользователями, которые совершали эти действия. \
Посмотрите на соответствующие исключения списков друг из друга, чтобы определить пользователей с проблемной последовательностью событий.

In [16]:
player_screen_copy = player_screen.copy()

In [17]:
player_screen_copy['flag_pause'] = np.where(player_screen_copy['action_id'] == 'playback_pause',1,0)
player_screen_copy['flag_unpause'] = np.where(player_screen_copy['action_id'] == 'playback_unpause',1,0)                                          
player_screen_copy['flag_start'] = np.where(player_screen_copy['action_id'] == 'playback_start',1,0)

player_screen_copy_gr = player_screen_copy.groupby('account_id').agg(cnt_pause = ('flag_pause','sum'),\
                                                         cnt_unpause = ('flag_unpause','sum'), \
                                                        cnt_start = ('flag_start','sum')).reset_index()
acc_df = player_screen_copy_gr[((player_screen_copy_gr['cnt_pause'] == 0)&(player_screen_copy_gr['cnt_unpause']>0))
|((player_screen_copy_gr['cnt_pause']>0) & (player_screen_copy_gr['cnt_start']==0))]


acc = acc_df['account_id']

df_pause = player_screen_copy_gr[((player_screen_copy_gr['cnt_pause'] == 0)&(player_screen_copy_gr['cnt_unpause']>0))]['account_id'].to_list()
df_start = player_screen_copy_gr[((player_screen_copy_gr['cnt_pause']>0) & (player_screen_copy_gr['cnt_start']==0))]['account_id'].to_list()

df_browser = player_screen_copy[player_screen_copy['account_id'].isin(acc)].groupby('user_browser')\
.agg(cnt_browser = ('user_browser', 'count')).reset_index()['user_browser'].to_list()
df_types = player_screen_copy[player_screen_copy['account_id'].isin(acc)].groupby('device_type')\
.agg(cnt_type = ('device_type', 'count')).reset_index()['device_type'].to_list()


print(f'Пользователи, которые снимали видео с паузы, но не ставили на паузу: {df_pause}')
print(f'Пользователи, которые ставили видео на паузу, но не включали проигрыватель: {df_start}')
print()
print(f'Чаще всего использовались браузеры {df_browser[0]} и тип устройства {df_types[0]}')




Пользователи, которые снимали видео с паузы, но не ставили на паузу: [102345, 102918, 103050]
Пользователи, которые ставили видео на паузу, но не включали проигрыватель: [102669, 103052]

Чаще всего использовались браузеры Chrome и тип устройства desktop_web


### Задача 3

1. Изучите пользователей, у которых есть только одно событие. Какой вид события встречается чаще всего? Какие из встречающихся событий вы бы посчитали ошибочными?
2. Изучите пользователей, у которых есть только два события. Какие из их последовательностей событий вы бы посчитали ошибочными?

In [18]:
cnt = player_screen.groupby('account_id').count()['server_time'].reset_index()
cnt_action = cnt[cnt['server_time']==1]['account_id'].tolist()

player_screen[player_screen['action_id'].isin(cnt_action)].groupby('action_id')\
.agg(cnt_12=('account_id','count')).reset_index()


,action_id,cnt_12


In [20]:
cnt = player_screen.groupby('account_id').count()['server_time'].reset_index()
cnt_action = cnt[cnt['server_time']==1]['account_id'].to_list()

# Фильтруем по account_id, а не по action_id
player_screen[player_screen['account_id'].isin(cnt_action)].groupby('action_id')\
.agg(cnt_12=('account_id','count')).reset_index()

,action_id,cnt_12
0,click,2
1,playback_start,2
2,playback_stop,3
